# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taqadussana/ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: "What Predicts Health?" — Random Forest feature importance (ML Appendix)

**The claim:** Average Position (43%), Impressions (32%), and Scroll Depth (15%) are the top
predictors of health_score, per a Random Forest's feature importance.

**My methodology question:** Health Score is explicitly defined earlier in the paper as
`Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)` — a formula
built directly from these same features. So the top 4 features by importance (position,
impressions, scroll depth, CTR) are literally the four ingredients of the label itself. Doesn't
that mean the model is mostly re-deriving its own construction formula rather than discovering
an independent predictive relationship? The paper does flag this honestly ("does not imply
external causation"), and I think that caution is exactly right — my question is whether this
finding would be more useful reframed around a genuinely external, independently observed
outcome (like future impression growth), rather than health_score itself, since a label built
from your own features will always show those features as "important" almost by construction.

### Finding 2: The Freshness Multiplier — 3.2x health boost, 57x impressions from refresh

**The claim:** "365+ day content that was refreshed within 30 days shows 3.2x health boost
(from 10.7 to 34.5) and 57x more impressions (from 71 to 4,039)... refresh timing is one of the
strongest measured levers available."

**My methodology question:** This compares refreshed old pages against (presumably)
non-refreshed old pages — but the paper doesn't say how pages got selected for refresh. If
editors chose which old pages to refresh (rather than refreshing being effectively random),
they likely picked pages that already showed some recovery potential, prior traffic history,
or strategic importance — meaning part of the 3.2x/57x gap could reflect *which* pages got
chosen, not the refresh itself. This is exactly the selection-bias check the claim ladder
teaches: "if the treated group was CHOSEN, part of the gap is the choosing, not the treatment."
I'd ask: was refresh assignment closer to random, or editor-selected? Without that detail, the
honest framing is closer to "refreshed pages recovered more" (observed/directional) rather than
"refresh timing is one of the strongest levers" (which reads closer to a causal claim).

Both questions are asked in the spirit the paper itself sets — it already self-flags uncertainty
in several places (small-n warnings, "descriptive not causal" caveats), and I'm applying that
same standard consistently rather than picking on weaker points.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit

url = "https://raw.githubusercontent.com/taqadussana/ML/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

features = ["avg_position", "impressions_90d", "sessions_90d", "content_age_days",
            "days_since_last_update", "ctr", "engagement_rate", "word_count"]
features = [f for f in features if f in df.columns]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def run_split(splitter, X, y, groups=None):
    if groups is not None:
        train_idx, test_idx = next(splitter.split(X, y, groups))
    else:
        train_idx, test_idx = next(splitter.split(X, y))
    rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = rf.predict_proba(X.iloc[test_idx])[:, 1]
    y_test = y.iloc[test_idx].values
    return {k: precision_at_k(scores, y_test, k) for k in (20, 50)}

# BEFORE: random split (naive, does not respect client grouping)
random_split = ShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
random_result = run_split(random_split, X, y)

# AFTER: client-grouped split (honest — entire clients held out)
grouped_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
grouped_result = run_split(grouped_split, X, y, groups)

print("Before/after — random split vs client-grouped split\n")
print(f"{'K':<6}{'Random split':<16}{'Grouped split':<16}{'Gap':<10}")
for k in (20, 50):
    gap = random_result[k] - grouped_result[k]
    print(f"{k:<6}{random_result[k]:<16.3f}{grouped_result[k]:<16.3f}{gap:<10.3f}")

Before/after — random split vs client-grouped split

K     Random split    Grouped split   Gap       
20    0.950           0.550           0.400     
50    0.940           0.580           0.360     


Under a random split, Precision@20 was 0.950 and Precision@50 was 0.940 — but once I switched
to a client-grouped split, both dropped sharply, to 0.550 and 0.580 respectively. That's a gap
of 0.400 and 0.360 — a large amount of what looked like model "skill" under the random split
was actually the model memorizing which clients tend to have declining pages, not learning a
pattern that generalizes to clients it's never seen.

This is a genuinely important finding for my own work: had I only reported the random-split
number (0.950), I would have badly overstated how well this model performs in the real
scenario it's meant for — flagging pages for editors at NEW clients the model has no history
with. The grouped-split number (0.550) is the one I should actually report and trust going
forward, and it matches the number I already used in my Week 5 comparison table.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Leakage attack checklist ===\n")

# 1. Confirm no label-derived columns in features
label_derived = ["trend_direction", "trend_pct", "is_declining_label"]
leaked_in_features = [f for f in features if f in label_derived]
print(f"1. Label-derived columns found in features: {leaked_in_features or 'NONE — clean'}")

# 2. Confirm no product flags used
product_flags = ["health_score", "priority_score", "action_type", "refresh_tier", "needs_ctr_fix", "is_quick_win"]
leaked_flags = [f for f in features if f in product_flags]
print(f"2. Product flags found in features: {leaked_flags or 'NONE — clean'}")

# 3. Deliberate leak test: add a label-derived column ON PURPOSE, watch the score jump
df["leaky_feature"] = df["trend_pct"]  # directly derived from the label
X_leaky = df[features + ["leaky_feature"]].replace([np.inf, -np.inf], np.nan).fillna(0)

leaky_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
leaky_result = run_split(leaky_split, X_leaky, y, groups)
clean_result = grouped_result  # from Section 2, already computed without the leak

print(f"\n3. Deliberate leak test (adding trend_pct as a feature):")
print(f"{'K':<6}{'Clean (no leak)':<18}{'With deliberate leak':<22}")
for k in (20, 50):
    print(f"{k:<6}{clean_result[k]:<18.3f}{leaky_result[k]:<22.3f}")

=== Leakage attack checklist ===

1. Label-derived columns found in features: NONE — clean
2. Product flags found in features: NONE — clean

3. Deliberate leak test (adding trend_pct as a feature):
K     Clean (no leak)   With deliberate leak  
20    0.550             1.000                 
50    0.580             1.000                 


No label-derived columns or product flags are present in my real feature set (checks 1 and 2
both came back clean). To prove my test harness actually catches leakage, I deliberately added
`trend_pct` — the exact column my label is derived from — as a feature.

As expected, Precision@K jumped from 0.550/0.580 (clean) to a perfect 1.000/1.000 with the
leak added. A perfect score is itself the confession — no real-world model is ever perfectly
correct on unseen data, so hitting 1.000 means the model wasn't predicting decline, it was
just reading the answer back from a column that already encodes it. This confirms my
evaluation setup is sensitive enough to catch real leakage when it's present, and gives me
confidence that my actual reported number (0.550/0.580, clean) is honest and not hiding a
similar issue.

The leaky column was removed immediately after this test and never used in Section 2 or in my
Week 5 reported results.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest prior claim (from Week 5):** "The Random Forest beats the baseline by a wide
margin at both cutoffs — nearly triple the precision at K=20."

**Rewritten in safe language:** "On a client-grouped, held-out test split, the Random Forest
showed a measured Precision@20 of 0.550 versus the baseline's 0.200 — a directional advantage
that held at Precision@50 as well (0.580 vs 0.320). This is decision-support evidence from one
train/test split on one month's data, not a claim that this model will outperform the baseline
in every client or every time period going forward."

The rewrite keeps the real numbers but removes language that implied certainty or permanence
("beats," "wide margin") in favor of language that ties the claim to exactly the evidence that
supports it — one split, one dataset, observed and measured, not guaranteed.

**A second claim worth tightening, in light of this week's own audit:** my Week 5 number
itself (0.550/0.580) initially looked stable, but this week's random-vs-grouped comparison
showed a random split alone would have overstated it to 0.950/0.940 — nearly double. That's a
reminder to always name which split produced a number, since the same model on the same data
can look dramatically different depending on validation design.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.